# Business dashboard: Brand Manager (Brand#32)

Visual answers to the Brand Manager questions (Q1-Q4) and monitoring, read only from the Gold layer.
Run after `06_run_pipeline` (or `05_run_member2`). The same queries power `dashboards/brand_manager.lvdash.json`.

Queries are identical to `sql/dashboard/*.sql` (checked by the test suite).

In [ ]:
%pip install -q plotly

In [ ]:
# Same SQL as sql/dashboard/*.sql (kept in sync by tests/test_dashboard.py).
QUERIES = {
    'kpi_brand32': "-- title: Brand#32 headline KPIs (Q1, Q2, Q4)\n-- revenue = net of discount, excluding tax (see README \"Business definitions\")\nWITH b32 AS (\n    SELECT SUM(brand_revenue) AS brand_revenue,\n           COUNT(DISTINCT category) AS category_count\n    FROM bda_gold.brand_category_monthly\n    WHERE brand = 'Brand#32'\n),\nb32_categories AS (\n    SELECT SUM(m.brand_revenue) AS category_revenue\n    FROM bda_gold.brand_category_monthly m\n    WHERE m.category IN (\n        SELECT DISTINCT category FROM bda_gold.brand_category_monthly WHERE brand = 'Brand#32'\n    )\n),\nmargin AS (\n    SELECT COUNT(*) AS product_count,\n           AVG(average_list_unit_margin) AS average_list_unit_margin\n    FROM bda_gold.part_list_margin\n    WHERE brand = 'Brand#32'\n),\ndeviation AS (\n    SELECT COUNT(*) AS deviating_product_count FROM bda_gold.price_deviation_products\n)\nSELECT\n    ROUND(b32.brand_revenue, 2)                                    AS brand32_revenue,\n    ROUND(b32.brand_revenue / c.category_revenue * 100, 2)         AS brand32_share_pct,\n    b32.category_count                                             AS brand32_category_count,\n    m.product_count                                                AS brand32_product_count,\n    ROUND(m.average_list_unit_margin, 2)                           AS brand32_average_margin,\n    d.deviating_product_count                                      AS deviating_product_count\nFROM b32\nCROSS JOIN b32_categories c\nCROSS JOIN margin m\nCROSS JOIN deviation d",
    'q1_share_by_category': "-- title: Q1 - Brand#32 share within each category it belongs to\n-- fair_share_pct = 100 / number of brands in the category (equal split benchmark)\nWITH b32 AS (\n    SELECT category,\n           SUM(brand_revenue)    AS brand32_revenue,\n           SUM(category_revenue) AS category_revenue\n    FROM bda_gold.brand_category_monthly\n    WHERE brand = 'Brand#32'\n    GROUP BY category\n),\nbrands AS (\n    SELECT category, COUNT(DISTINCT brand) AS brand_count\n    FROM bda_gold.brand_category_monthly\n    GROUP BY category\n)\nSELECT\n    b.category,\n    ROUND(b.brand32_revenue, 2)                                         AS brand32_revenue,\n    ROUND(b.category_revenue, 2)                                        AS category_revenue,\n    ROUND(b.brand32_revenue / NULLIF(b.category_revenue, 0) * 100, 2)   AS brand32_share_pct,\n    ROUND(100.0 / n.brand_count, 2)                                     AS fair_share_pct\nFROM b32 b\nJOIN brands n ON b.category = n.category\nORDER BY brand32_share_pct DESC",
    'q2_margin_by_brand': "-- title: Q2 - average list-price margin per brand (Brand#32 highlighted)\n-- per product: average over supplier offers; per brand: equal-weight average over products\nSELECT\n    brand,\n    COUNT(*)                                  AS product_count,\n    COUNT(average_list_unit_margin)           AS products_with_known_margin,\n    ROUND(AVG(average_list_unit_margin), 2)   AS average_list_unit_margin,\n    CASE WHEN brand = 'Brand#32' THEN 'Brand#32' ELSE 'Other brands' END AS highlight\nFROM bda_gold.part_list_margin\nGROUP BY brand\nORDER BY average_list_unit_margin DESC",
    'q3_category_leaders': "-- title: Q3 - category leader (largest all-time share) per category\nWITH totals AS (\n    SELECT category, brand, SUM(brand_revenue) AS brand_revenue\n    FROM bda_gold.brand_category_monthly\n    GROUP BY category, brand\n),\nshares AS (\n    SELECT category, brand,\n           brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share\n    FROM totals\n),\nranked AS (\n    SELECT *, RANK() OVER (PARTITION BY category ORDER BY market_share DESC) AS category_rank\n    FROM shares\n)\nSELECT\n    category,\n    brand,\n    ROUND(market_share * 100, 4) AS market_share_pct,\n    CONCAT(brand, ' / ', category) AS label\nFROM ranked\nWHERE category_rank = 1\nORDER BY market_share_pct DESC\nLIMIT 15",
    'q3_leader_stability': "-- title: Q3 - is the overall leader's share stable? (monthly share and rank, complete months only)\n-- boundary months (first and last) are excluded as potentially incomplete, same rule as 02_gold_layer\nWITH totals AS (\n    SELECT category, brand, SUM(brand_revenue) AS brand_revenue\n    FROM bda_gold.brand_category_monthly\n    GROUP BY category, brand\n),\nshares AS (\n    SELECT category, brand,\n           brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share\n    FROM totals\n),\nleader AS (\n    SELECT category, brand\n    FROM shares\n    ORDER BY market_share DESC, category, brand\n    LIMIT 1\n),\nbounds AS (\n    SELECT MIN(month_start) AS min_month, MAX(month_start) AS max_month\n    FROM bda_gold.brand_category_monthly\n),\nmonthly AS (\n    SELECT m.month_start, m.category, m.brand, m.market_share,\n           RANK() OVER (PARTITION BY m.month_start, m.category ORDER BY m.market_share DESC) AS monthly_rank\n    FROM bda_gold.brand_category_monthly m\n    JOIN leader l ON m.category = l.category\n    CROSS JOIN bounds b\n    WHERE m.market_share IS NOT NULL\n      AND m.month_start > b.min_month\n      AND m.month_start < b.max_month\n)\nSELECT\n    m.month_start,\n    m.brand,\n    m.category,\n    ROUND(m.market_share * 100, 2) AS market_share_pct,\n    m.monthly_rank\nFROM monthly m\nJOIN leader l ON m.brand = l.brand\nORDER BY m.month_start",
    'q4_price_deviation_distribution': "-- title: Q4 - distribution of sale-line price deviation vs. the 15% threshold\n-- deviation = |actual unit price - list price| / list price, bucketed by whole percent\nSELECT\n    CAST(FLOOR(price_deviation_ratio * 100) AS INT) AS deviation_pct_bucket,\n    COUNT(*)                                        AS sale_lines,\n    CASE WHEN price_deviation_ratio > 0.15 THEN 'above 15% threshold'\n         ELSE 'within threshold' END                AS threshold_status\nFROM bda_gold.brand_sales\nGROUP BY 1, 3\nORDER BY deviation_pct_bucket",
    'monitoring_brand32_trend': "-- title: Monitoring - Brand#32 revenue and share across its categories, complete months\nSELECT\n    month_start,\n    ROUND(SUM(brand_revenue), 2)                                    AS brand32_revenue,\n    ROUND(SUM(brand_revenue) / NULLIF(SUM(category_revenue), 0) * 100, 3) AS brand32_share_pct\nFROM bda_gold.brand_monitoring\nWHERE brand = 'Brand#32'\n  AND is_complete_month\nGROUP BY month_start\nORDER BY month_start",
    'monitoring_latest_change': "-- title: Monitoring - Brand#32 share change in the latest comparable month vs. the -5 pp alert threshold\nSELECT\n    month_start,\n    category,\n    ROUND(market_share * 100, 2)              AS market_share_pct,\n    ROUND(share_change_percentage_points, 2)  AS share_change_pp,\n    -5.0                                      AS alert_threshold_pp,\n    CASE WHEN share_change_percentage_points <= -5 THEN 'ALERT' ELSE 'ok' END AS alert_status\nFROM bda_gold.brand_monitoring\nWHERE brand = 'Brand#32'\n  AND is_comparable_month\n  AND month_start = (\n      SELECT MAX(month_start) FROM bda_gold.brand_monitoring\n      WHERE brand = 'Brand#32' AND is_comparable_month\n  )\nORDER BY share_change_pp",
    'monitoring_change_explanation': "-- title: Monitoring - why did the share move? brand effect vs. category effect (10 largest moves)\nSELECT\n    month_start,\n    category,\n    ROUND(brand_revenue_effect_pp, 3)    AS brand_effect_pp,\n    ROUND(category_revenue_effect_pp, 3) AS category_effect_pp,\n    ROUND(actual_share_change_pp, 3)     AS actual_change_pp\nFROM bda_gold.brand_share_change_explanation\nWHERE brand = 'Brand#32'\n  AND month_start = (\n      SELECT MAX(month_start) FROM bda_gold.brand_share_change_explanation WHERE brand = 'Brand#32'\n  )\nORDER BY ABS(actual_share_change_pp) DESC\nLIMIT 10",
    'alert_status': "-- title: Alert - Brand#32 share drop >= 5 pp in the latest complete month (same logic as sql/brand_share_alert.sql)\nWITH latest AS (\n    SELECT MAX(month_start) AS month_start\n    FROM bda_gold.brand_monitoring WHERE is_complete_month\n)\nSELECT\n    l.month_start,\n    COUNT(CASE WHEN m.is_comparable_month AND m.share_change_percentage_points <= -5 THEN 1 END) AS affected_category_count,\n    COUNT(CASE WHEN NOT m.is_comparable_month THEN 1 END) AS unavailable_category_count,\n    COUNT(m.category) AS monitored_category_count\nFROM latest l\nLEFT JOIN bda_gold.brand_monitoring m\n    ON m.month_start = l.month_start AND m.brand = 'Brand#32'\nGROUP BY l.month_start",
    'validation_summary': "-- title: Data quality - latest validation run, checks passed per phase\nWITH latest AS (\n    SELECT run_id FROM bda_gold.validation_results\n    ORDER BY checked_at DESC\n    LIMIT 1\n)\nSELECT\n    v.phase,\n    COUNT(*)                                        AS checks,\n    SUM(CASE WHEN v.passed THEN 1 ELSE 0 END)       AS passed,\n    SUM(CASE WHEN NOT v.passed THEN 1 ELSE 0 END)   AS failed,\n    SUM(CASE WHEN NOT v.passed AND v.is_critical THEN 1 ELSE 0 END) AS critical_failures\nFROM bda_gold.validation_results v\nJOIN latest l ON v.run_id = l.run_id\nGROUP BY v.phase\nORDER BY CASE v.phase WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2\n                      WHEN 'reconciliation' THEN 3 WHEN 'gold' THEN 4 ELSE 5 END",
}

In [ ]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

dbutils.widgets.text("catalog", "workspace", "Target catalog")
CATALOG = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(f"invalid catalog: {CATALOG!r}")
spark.sql(f"USE CATALOG {CATALOG}")

# palette: one highlight hue, gray context, reserved alert red (see README > Visual conventions)
HIGHLIGHT, CONTEXT, ALERT, GOOD = "#2a78d6", "#b5b4ad", "#e34948", "#1baf7a"
LAYOUT = dict(template="simple_white", font=dict(size=13), margin=dict(l=40, r=20, t=60, b=40), height=420)


def q(name: str) -> pd.DataFrame:
    """Run one dashboard dataset (same SQL as dashboards/brand_manager.lvdash.json)."""
    pdf = spark.sql(QUERIES[name]).toPandas()
    for c in pdf.columns:  # Spark DECIMAL -> float for plotting
        if pdf[c].dtype == object and len(pdf) and pdf[c].map(lambda v: hasattr(v, "as_tuple")).all():
            pdf[c] = pdf[c].astype(float)
    return pdf


def kpis(items):
    tiles = "".join(
        f"<div style='padding:12px 18px;margin:4px;border:1px solid #ddd;border-radius:8px;min-width:200px'>"
        f"<div style='color:#52514e;font-size:13px'>{label}</div>"
        f"<div style='font-size:28px;font-weight:600'>{value}</div></div>" for label, value in items)
    displayHTML(f"<div style='display:flex;flex-wrap:wrap;font-family:sans-serif'>{tiles}</div>")

print(f"catalog={CATALOG}; {len(QUERIES)} datasets loaded")

## Headline numbers

In [ ]:
k = q("kpi_brand32").iloc[0]
kpis([
    ("Q1 Brand#32 net revenue", f"${k.brand32_revenue/1e9:,.2f} B"),
    ("Q1 share across its categories", f"{k.brand32_share_pct:.2f} %"),
    ("Categories with Brand#32 products", f"{int(k.brand32_category_count)}"),
    ("Q2 average list-price margin", f"${k.brand32_average_margin:,.2f}"),
    ("Q2 Brand#32 products", f"{int(k.brand32_product_count):,}"),
    ("Q4 products with deviation > 15%", f"{int(k.deviating_product_count):,}"),
])

## Q1. Brand#32 revenue and share within its categories
Brand#32 has products in every category, so the answer is a distribution. The dashed line is the equal-split benchmark (100% / number of brands).

In [ ]:
df = q("q1_share_by_category").sort_values("brand32_share_pct", ascending=False)
fig = px.bar(df, x="category", y="brand32_share_pct", hover_data=["brand32_revenue", "category_revenue"],
             title="Brand#32 share of category revenue, % (sorted)", color_discrete_sequence=[HIGHLIGHT])
fig.add_hline(y=df.fair_share_pct.mean(), line_dash="dash", line_color="#52514e",
              annotation_text=f"fair share {df.fair_share_pct.mean():.1f}%")
fig.update_layout(**LAYOUT, xaxis=dict(showticklabels=False, title=f"{len(df)} categories"), yaxis_title="share, %")
fig.show()
display(pd.concat([df.head(5), df.tail(5)]))

## Q2. Average list-price margin (list price minus supply cost)
Per product: average over its supplier offers (no double counting across suppliers); per brand: equal-weight average over products.

In [ ]:
df = q("q2_margin_by_brand").sort_values("average_list_unit_margin", ascending=False)
fig = px.bar(df, x="brand", y="average_list_unit_margin", color="highlight",
             color_discrete_map={"Brand#32": HIGHLIGHT, "Other brands": CONTEXT},
             title="Average list-price margin by brand", hover_data=["product_count"])
fig.update_layout(**LAYOUT, yaxis_title="margin per unit, $", xaxis_title=None, legend_title=None)
lo, hi = df.average_list_unit_margin.min(), df.average_list_unit_margin.max()
fig.update_yaxes(range=[lo * 0.98, hi * 1.01])
fig.show()

## Q3. Which brand has the largest share within its category, and is it stable?
Stability rule (project assumption): monthly share range <= 5 pp over complete months.

In [ ]:
df = q("q3_category_leaders").sort_values("market_share_pct")
fig = px.bar(df, x="market_share_pct", y="label", orientation="h", title="Category leaders by all-time share, % (top 15)",
             color_discrete_sequence=[CONTEXT])
fig.update_traces(marker_color=[HIGHLIGHT if i == len(df) - 1 else CONTEXT for i in range(len(df))])
fig.update_layout(**{**LAYOUT, "height": 520}, xaxis_title="share, %", yaxis_title=None)
fig.show()

s = q("q3_leader_stability")
brand, category = s.brand.iloc[0], s.category.iloc[0]
rng = s.market_share_pct.max() - s.market_share_pct.min()
verdict = "STABLE" if rng <= 5 else "UNSTABLE"
fig = go.Figure(go.Scatter(x=s.month_start, y=s.market_share_pct, mode="lines", line=dict(color=HIGHLIGHT, width=2), name="monthly share"))
fig.add_hrect(y0=s.market_share_pct.min(), y1=s.market_share_pct.max(), fillcolor=HIGHLIGHT, opacity=0.08, line_width=0)
fig.update_layout(**LAYOUT, title=f"{brand} in {category}: range {rng:.2f} pp over {len(s)} months -> {verdict}",
                  yaxis_title="share, %")
fig.show()
fig = px.line(s, x="month_start", y="monthly_rank", title=f"{brand}: monthly rank in {category} (1 = leader)")
fig.update_traces(line_color=CONTEXT)
fig.update_layout(**LAYOUT, yaxis=dict(autorange="reversed", title="rank"))
fig.show()

## Q4. Products whose sale price differs from list price by more than 15%

In [ ]:
df = q("q4_price_deviation_distribution")
fig = px.bar(df, x="deviation_pct_bucket", y="sale_lines", color="threshold_status",
             color_discrete_map={"within threshold": CONTEXT, "above 15% threshold": ALERT},
             title="Sale lines by price deviation from list price")
fig.add_vline(x=15, line_dash="dash", line_color=ALERT, annotation_text="15% threshold")
fig.update_layout(**LAYOUT, xaxis=dict(title="deviation, % (bucket)", range=[-1, 20]), yaxis_title="sale lines", legend_title=None)
fig.show()
print("max deviation bucket observed:", int(df.deviation_pct_bucket.max()), "%")

## Monitoring: Brand#32 share over time and the -5 pp alert

In [ ]:
t = q("monitoring_brand32_trend")
fig = px.line(t, x="month_start", y="brand32_share_pct", title="Brand#32 share across its categories, % (complete months)")
fig.update_traces(line_color=HIGHLIGHT)
fig.update_layout(**LAYOUT, yaxis_title="share, %")
fig.show()

c = q("monitoring_latest_change").sort_values("share_change_pp")
month = str(c.month_start.iloc[0])[:7]
fig = px.bar(c, x="category", y="share_change_pp", title=f"{month}: Brand#32 share change by category, pp",
             color=c.share_change_pp.map(lambda v: "ALERT" if v <= -5 else ("loss" if v < 0 else "gain")),
             color_discrete_map={"ALERT": ALERT, "loss": CONTEXT, "gain": HIGHLIGHT})
fig.add_hline(y=-5, line_dash="dash", line_color=ALERT, annotation_text="alert threshold -5 pp")
fig.update_layout(**LAYOUT, xaxis=dict(showticklabels=False, title=f"{len(c)} categories"), yaxis_title="pp", legend_title=None)
fig.show()

a = q("alert_status").iloc[0]
kpis([("Alert month", str(a.month_start)[:7]),
      ("Categories with drop >= 5 pp", int(a.affected_category_count)),
      ("Categories monitored", int(a.monitored_category_count)),
      ("Categories without comparison", int(a.unavailable_category_count))])

### Why did the share move? Brand effect vs. category effect
share change = brand revenue effect + category revenue effect (identity validated in `03_validation`).

In [ ]:
e = q("monitoring_change_explanation")
long = e.melt(id_vars="category", value_vars=["brand_effect_pp", "category_effect_pp"], var_name="effect", value_name="pp")
fig = px.bar(long, x="category", y="pp", color="effect", barmode="relative",
             color_discrete_map={"brand_effect_pp": HIGHLIGHT, "category_effect_pp": "#eb6834"},
             title="10 largest Brand#32 share moves, decomposed")
fig.add_scatter(x=e.category, y=e.actual_change_pp, mode="markers", marker=dict(color="#0b0b0b", size=9), name="actual change")
fig.update_layout(**LAYOUT, yaxis_title="pp", xaxis_title=None, legend_title=None)
fig.show()

## Data quality: latest validation run

In [ ]:
v = q("validation_summary")
fig = go.Figure([go.Bar(name="passed", x=v.phase, y=v.passed, marker_color=GOOD),
                 go.Bar(name="failed", x=v.phase, y=v.failed, marker_color=ALERT)])
fig.update_layout(**LAYOUT, barmode="stack", title=f"{int(v.passed.sum())}/{int(v.checks.sum())} checks passed", yaxis_title="checks")
fig.show()
display(v)